In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from llm_judge import create_log_judge_agent, format_judge_prompt

In [3]:
import json

with open("evals_run_2026_08_05_results.json", "r") as f:
    results = json.load(f)

print(f"Loaded {len(results)} results.")

Loaded 27 results.


In [4]:
judge = create_log_judge_agent()

In [5]:
row = results[0]

In [6]:
prompt = format_judge_prompt(row)

In [7]:
eval_result = await judge.run(prompt)
eval_output = eval_result.output

In [8]:
print(eval_output.label)
print(eval_output.reasoning)

good
The user asked how to set up an LLM as a judge. The agent provided a step-by-step guide with code examples. The tools used were search and get_file, which are appropriate for retrieving documentation and examples. The response accurately reflects the content of the documentation, covering installation, API key setup, dataset creation, prompt definition, running the judge, and evaluating its quality. The code snippets are relevant and correctly formatted. The response directly answers the user's question and does not include any off-topic information or hallucinations.


In [9]:
async def run_eval(row):
    prompt = format_judge_prompt(row)
    eval_result = await judge.run(prompt)
    eval_output = eval_result.output
    return eval_output

In [10]:
from tqdm.auto import tqdm

In [12]:
eval_results = []

for row in tqdm(results):
    eval_output = await run_eval(row)
    eval_results.append({
        'row': row,
        'label': eval_output.label,
        'reasoning': eval_output.reasoning
    })

  0%|          | 0/27 [00:00<?, ?it/s]

In [13]:
rows = []

for result in eval_results:
    row = {
        'question': result['row']['input']['question'],
        'llm_answer': result['row']['rag_response']['answer'],
        'human_label': result['row']['label'],
        'human_comment': result['row']['comments'],
        'llm_label': result['label'],
        'llm_reasoning': result['reasoning']
    }
    rows.append(row)

In [14]:
import pandas as pd
df_rows = pd.DataFrame(rows)

In [16]:
df_rows

,question,llm_answer,human_label,human_comment,llm_label,llm_reasoning
0,How do I set up LLM as a judge?,"To set up an LLM as a judge, you will define c...",good,,good,The agent correctly understood the user's ques...
1,I want to evaluate my chatbot responses,Evidently helps you evaluate chatbot responses...,good,,good,The user is asking about evaluating chatbot re...
2,How do I customize evaluation criteria for my ...,To customize evaluation criteria for your use ...,good,,good,The user is asking about customizing evaluatio...
3,What metrics are available?,Evidently offers a wide range of metrics categ...,good,,good,The agent correctly identified the user's ques...
4,How do I measure text length?,"To measure text length, you can use the `TextL...",good,,good,The user asked how to measure text length. The...
5,What's the difference between metrics and desc...,Descriptors and metrics serve different purpos...,good,,good,The user asked for the difference between metr...
6,How do I add a scoring function?,You can add a scoring function by implementing...,good,,good,The user asked how to add a scoring function. ...
7,How do I export a report as HTML?,You can export a report as an HTML file using ...,good,,good,The user asked how to export a report as HTML....
8,How do I save my results as a PDF?,"Based on the documentation, there is no direct...",good,,good,The user is asking about saving results as a P...
9,How do I add a panel to my dashboard?,"To add a panel to your dashboard, you must fir...",good,,bad,The user asked how to add a panel to their das...


In [18]:
df_rows.to_csv("evals_run_2026_08_05_judge_results.csv")

In [19]:
df_rows.llm_label.value_counts()

llm_label
good    25
bad      2
Name: count, dtype: int64

In [20]:
df_rows.human_label.value_counts()

human_label
good    26
bad      1
Name: count, dtype: int64

In [21]:
(df_rows.llm_label == df_rows.human_label).mean()

np.float64(0.8888888888888888)

In [22]:
df_disagreement = df_rows[df_rows.llm_label != df_rows.human_label]

In [23]:
df_disagreement

,question,llm_answer,human_label,human_comment,llm_label,llm_reasoning
9,How do I add a panel to my dashboard?,"To add a panel to your dashboard, you must fir...",good,,bad,The user asked how to add a panel to their das...
14,How do I create a new project?,"To create a new project, you can use either th...",good,,bad,The user asked how to create a new project. Th...
26,How do I set up a Grafana dashboard for model ...,To set up a Grafana dashboard for model monito...,bad,Hallucination,good,The user is asking about setting up a Grafana ...
